# Training Process

## Get the data

In [1]:
import pickle
import math
import numpy as np
import tensorflow as tf

with open('Data/out/pickle/mel_spec.pkl', 'rb') as file:
    
    (X_melspec_train, X_melspec_test, X_melspec_valid, y_melspec_train, y_melspec_test, y_melspec_valid) = pickle.load(file)


In [2]:
X_train = np.array([spec.flatten() for spec in X_melspec_train])[:10000] # small data only for demonstration
X_test = np.array([spec.flatten() for spec in X_melspec_test])[:10000]
X_valid = np.array([spec.flatten() for spec in X_melspec_test])[:10000]

y_train = y_melspec_train[:10000]
y_test = y_melspec_test[:10000]
y_valid = y_melspec_valid[:10000]

## Create the network architecture

In [3]:
input_shape = X_train[0].shape

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.Dense(88, activation='sigmoid'),
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 88)                35288     
Total params: 35,288
Trainable params: 35,288
Non-trainable params: 0
_________________________________________________________________


## Model Parameters

In [4]:
BATCH_SIZE = 128
EPOCHS = 100

## Choose optimizer and loss

In [5]:
model.compile(optimizer='rmsprop', loss='mse', metrics=['accuracy'])

## Set up Early Stopping

In [6]:
# See Lecture 11 - NeuralNetworks slide 54
# https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
early_stop_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
# This callback will stop the training when there is no improvement in
# the validation loss for three consecutive epochs.

## Set up model checkpoints

In [7]:
CHECKPOINT_PATH = "training-example/checkpoints/"
CHECKPOINT_NAME = "example_cp-"

SAVE_AFTER_EPOCHS = 1

save_frequency = BATCH_SIZE * SAVE_AFTER_EPOCHS # save model each epoch

# Create a callback that saves the model's weights
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=CHECKPOINT_PATH+CHECKPOINT_NAME+"{epoch:04d}.ckpt",
                                                 save_weights_only=True,
                                                 verbose=1,
                                                 save_freq=save_frequency)

In [8]:
history = model.fit(X_train, y_train,
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop_callback, checkpoint_callback],
                    validation_data=(X_test, y_test))

Epoch 1/100
79/79 [==============================] - 0s 4ms/step - loss: 0.0646 - accuracy: 0.0220 - val_loss: 0.0420 - val_accuracy: 0.0239
Epoch 2/100
31/79 [==========>...................] - ETA: 0s - loss: 0.0412 - accuracy: 0.0219
Epoch 00002: saving model to training-example/checkpoints\example_cp-0002.ckpt
79/79 [==============================] - 0s 4ms/step - loss: 0.0404 - accuracy: 0.0240 - val_loss: 0.0379 - val_accuracy: 0.0248
Epoch 3/100
79/79 [==============================] - 0s 2ms/step - loss: 0.0378 - accuracy: 0.0305 - val_loss: 0.0363 - val_accuracy: 0.0330
Epoch 4/100
 1/79 [..............................] - ETA: 0s - loss: 0.0361 - accuracy: 0.0234
Epoch 00004: saving model to training-example/checkpoints\example_cp-0004.ckpt
79/79 [==============================] - 0s 3ms/step - loss: 0.0363 - accuracy: 0.0397 - val_loss: 0.0351 - val_accuracy: 0.0452
Epoch 5/100
41/79 [==============>...............] - ETA: 0s - loss: 0.0356 - accuracy: 0.0431
Epoch 00005: savi

In [9]:
history.history

{'loss': [0.06457244604825974,
  0.04038843512535095,
  0.03781896084547043,
  0.03634650632739067,
  0.03521726652979851,
  0.03428273648023605,
  0.03349912911653519,
  0.03281524404883385,
  0.03219517692923546,
  0.03167274594306946,
  0.031166113913059235,
  0.03071979060769081,
  0.030310289934277534,
  0.02994862012565136,
  0.029625415802001953,
  0.029333414509892464,
  0.029083922505378723,
  0.028841396793723106,
  0.02862904593348503,
  0.02844429947435856,
  0.028263341635465622,
  0.028112243860960007,
  0.027982082217931747,
  0.02784423902630806,
  0.027728665620088577,
  0.027621304616332054,
  0.027519317343831062,
  0.02743285894393921,
  0.027350781485438347,
  0.027267714962363243,
  0.0271940678358078,
  0.027121365070343018,
  0.027059540152549744,
  0.02700292132794857,
  0.02695363759994507,
  0.026905450969934464,
  0.026849178597331047,
  0.02680477686226368,
  0.026760561391711235,
  0.026713958010077477,
  0.026681942865252495,
  0.02664562314748764,
  0.02

## Select best model

In [10]:
def get_best_model(history, model):
    min_loss_idx = 0
    for i in range(len(history)):
        if history[i] < history[min_loss_idx]:
            min_loss_idx = i
    #CHECKPOINT_NAME = f'example_cp-{epoch:04d}.ckpt'
    model.load_weights(CHECKPOINT_PATH+CHECKPOINT_NAME+f"{min_loss_idx:04d}.ckpt")
    return model

In [11]:
best_model = get_best_model(history.history['val_loss'], model)

## Report Validation Set Measures

In [12]:
from  Utils import Evaluation

pred = best_model(X_valid)

print("Confusion Matrix", Evaluation.get_confusion_matrix(pred, y_valid))
print("Accuracy", Evaluation.get_metric(pred, y_valid, metric='accuracy'))
print("Precision", Evaluation.get_metric(pred, y_valid, metric='precision'))
print("Recall", Evaluation.get_metric(pred, y_valid, metric='recall'))

Confusion Matrix tf.Tensor(
[[848170  27300]
 [  4262    268]], shape=(2, 2), dtype=int32)
Accuracy 0.9641341
Precision 0.009721416
Recall 0.05916115
